# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results.

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone.


In [1]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV,
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:**

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [3]:
# 3. Load dataset from milestone 1
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    df = pd.read_csv("/content/drive/MyDrive/zillow_cleaned.csv")
else:
    df = pd.read_csv("/Users/seanamisano/Downloads/OMDS/DX603/DX603Milestone2/zillow_cleaned.csv")



print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: 75224 rows, 25 columns


In [4]:
# 4. Separate features and target, then train-test split and scale

X = df.drop(columns=['taxvaluedollarcnt'])
y = df['taxvaluedollarcnt']

# Keep only numeric columns (drop any remaining categorical)
X = X.select_dtypes(include=[np.number])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Standardize features using only training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train: {X_train_scaled.shape}, X_test: {X_test_scaled.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

X_train: (60179, 22), X_test: (15045, 22)
y_train: (60179,), y_test: (15045,)


In [5]:
# 5. Standardize X_train

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**.


In [7]:
# Add as many cells as you need

# Recommended from lectures: linear regression, decision tree, random forest

# Start with linear regression

lr = LinearRegression()
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

lr_scores = cross_val_score(lr, X_train_scaled, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
lr_mae = -lr_scores  # flip sign since sklearn returns negative MAE

print("=== Linear Regression ===")
print(f"Mean MAE: ${lr_mae.mean():,.2f}")
print(f"Std MAE:  ${lr_mae.std():,.2f}")


=== Linear Regression ===
Mean MAE: $171,386.51
Std MAE:  $1,027.34


In [8]:
# Decision Tree Regression with Repeated Cross-Validation
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=random_state)

dt_scores = cross_val_score(dt, X_train_scaled, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
dt_mae = -dt_scores

print("=== Decision Tree Regression ===")
print(f"Mean MAE: ${dt_mae.mean():,.2f}")
print(f"Std MAE:  ${dt_mae.std():,.2f}")

=== Decision Tree Regression ===
Mean MAE: $196,707.06
Std MAE:  $1,474.68


In [9]:
# Random Forest Regression with Repeated Cross-Validation
rf = RandomForestRegressor(random_state=random_state, n_jobs=-1)

rf_scores = cross_val_score(rf, X_train_scaled, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
rf_mae = -rf_scores

print("=== Random Forest Regression ===")
print(f"Mean MAE: ${rf_mae.mean():,.2f}")
print(f"Std MAE:  ${rf_mae.std():,.2f}")

# Ignore warnings, they are harmless and purely cosmetic

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


KeyboardInterrupt: 

### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

The best overall model is the random forest regression model. Followed by linear regression, then decision tree last. This is not surprising.

  - Decision tree with default settings essentially memorizes the data, overfitting on the training data and thus performs poorly on the testing data. We can improve its performance by reducing max_depth, preventing it from memorizing every data point at each node.

  - Linear regression is simple and overall pretty accurate. Fitting a single flat hyperplane through the data, it can't account for nonlinear relationships, so leaning more to underfitting. It generalizes consistently across training and testing data.

  - Random forest trains 100 decision trees on random subsets of data, then takes the averages of their predictions. This is optimal compared to our previous two models because each tree can still capture complex patterns missed in a linear regression, and averaging the many trees effectively dispels issues with overfitting and noise.

Stability:
    This measure followed the same trend as mean MAE. The random forest model had the lowest std due to ensemble averaging, discussed above. Therefore, this model had the overall best results, with both the lowest mean MAE and std.

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1.

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler`
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [ ]:
# Part 2: Feature Engineering
# Add 3 new engineered features to X_train and X_test

# Work with the unscaled DataFrames (X_train / X_test still have column names)
X_train_eng = X_train.copy()
X_test_eng = X_test.copy()

# Feature 1: Log of square footage (reduces right skew)
X_train_eng['log_sqft'] = np.log1p(X_train_eng['finishedsquarefeet12'])
X_test_eng['log_sqft'] = np.log1p(X_test_eng['finishedsquarefeet12'])

# Feature 2: Square feet per room (space efficiency)
X_train_eng['sqft_per_room'] = X_train_eng['finishedsquarefeet12'] / (X_train_eng['roomcnt'] + 1)
X_test_eng['sqft_per_room'] = X_test_eng['finishedsquarefeet12'] / (X_test_eng['roomcnt'] + 1)

# Feature 3: Bathrooms per bedroom (luxury indicator)
X_train_eng['bath_per_bed'] = X_train_eng['bathroomcnt'] / (X_train_eng['bedroomcnt'] + 1)
X_test_eng['bath_per_bed'] = X_test_eng['bathroomcnt'] / (X_test_eng['bedroomcnt'] + 1)

print(f"Original features: {X_train.shape[1]}")
print(f"After engineering: {X_train_eng.shape[1]}")
print(f"New features: log_sqft, sqft_per_room, bath_per_bed")

Original features: 22
After engineering: 25
New features: log_sqft, sqft_per_room, bath_per_bed


In [ ]:
# Re-scale with new features (fit only on training data)
scaler_eng = StandardScaler()
X_train_eng_scaled = scaler_eng.fit_transform(X_train_eng)
X_test_eng_scaled = scaler_eng.transform(X_test_eng)

print(f"X_train_eng_scaled: {X_train_eng_scaled.shape}")
print(f"X_test_eng_scaled:  {X_test_eng_scaled.shape}")

X_train_eng_scaled: (60179, 25)
X_test_eng_scaled:  (15045, 25)


In [ ]:
# Re-run all 3 models with engineered features (default params, same CV)

from sklearn.tree import DecisionTreeRegressor

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=random_state),
    'Random Forest': RandomForestRegressor(random_state=random_state, n_jobs=-1)
}

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

for name, model in models.items():
    scores = cross_val_score(model, X_train_eng_scaled, y_train, cv=cv,
                             scoring='neg_mean_absolute_error', n_jobs=-1)
    mae = -scores
    print(f"=== {name} (with engineered features) ===")
    print(f"Mean MAE: ${mae.mean():,.2f}")
    print(f"Std MAE:  ${mae.std():,.2f}\n")

# Again, ignore harmless warnings

=== Linear Regression (with engineered features) ===
Mean MAE: $171,437.91
Std MAE:  $1,025.34

=== Decision Tree (with engineered features) ===
Mean MAE: $197,204.23
Std MAE:  $1,495.81



/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


=== Random Forest (with engineered features) ===
Mean MAE: $151,568.37
Std MAE:  $942.61



### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




The engineered features did not improve any of the three models. This reflexts that the new features don't introduce new information into the models. Rather, they are mathematical transformations of existing features. Therefore, in terms of MAE and model performance, their main contribution is increasing multicollinearity with the features they are derived from. This conclusion is especially enforced by the poor peformance of linear regression in this second round, as this model is notoriously weakened by multicollinearity.

None of the three new features seemed to help.

  - 'log_sqft' is a monotonic transformation of 'finishedsquarefeet12', so it offers no new information.

  - 'sqft_per_room' and 'bath_per_bed' are ratios of existing features, which tree-based models can already compute through sequential splits.

Hypotheses about why the features didn't help:

  - Linear regression: hurt the most by the added features due to added multicollinearity. Linear regression solves for coefficients using matrix calculations, and similar columns make that matrix nearly singular (close to being non-invertible), destabilizing the coefficients

  - Decision Tree: Largely unaffected, because it simply picks the best performing feature and can ignore reduntant ones. However, adding more unhelpful features gives the model more opportunities to overfit on noise.

  - Random Forest: Most robust model of the three overall and in this case, since it randomly subsets features at each split, so reduntant features only occasionally impact the results.

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [ ]:
# Add as many cells as you need

# Part 3: Feature Selection

def forward_feature_selection(X, y, model,
                              scoring='neg_mean_absolute_error',
                              cv = 5,
                              tol=None,
                              max_features=None,
                              n_jobs=-1,
                              verbose=False
                             ):
    selected_features = []
    remaining_features = list(X.columns)
    best_scores = []
    previous_score = float('inf')



    best_feature_set = None
    best_score = float('inf')

    while remaining_features:
        scores = {}
        for feature in remaining_features:
            current_features = selected_features + [feature]

            # Compute the CV (negated MAE)
            cv_score = -cross_val_score(model, X[current_features], y,
                                        scoring=scoring, cv=cv, n_jobs=n_jobs
                                       ).mean()
            scores[feature] = cv_score

        # Select the feature that minimizes the CV score
        best_feature = min(scores, key=scores.get)
        current_score = scores[best_feature]

        # Check if the improvement is significant based on the tolerance (tol)
        if tol is not None and previous_score - current_score < tol:
            if verbose:
                print("Stopping early due to minimal improvement.")
            break

        # Add the best feature to the selected list and update score trackers
        selected_features.append(best_feature)
        best_scores.append(current_score)
        remaining_features.remove(best_feature)
        previous_score = current_score

        if verbose:
            print(f"Added Best Feature: {best_feature}, CV Score (MAE): {current_score:.4f}")

        # Update the best subset if the current score is better than the best so far
        if current_score < best_score:
            best_score = current_score
            best_feature_set = selected_features.copy()

        # Check if the maximum number of features has been reached
        if max_features is not None and len(selected_features) >= max_features:
            break

    return (selected_features, best_scores, best_feature_set,best_score)

In [ ]:
X_train_df = pd.DataFrame(X_train_eng_scaled, columns=X_train_eng.columns)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

# Feature Selection for Linear Regression
lr_feats, lr_scores, lr_best_set, lr_best_score = forward_feature_selection(X_train_df, y_train, LinearRegression(), scoring='neg_mean_absolute_error', cv = cv, verbose=True)

# Feature Selection for Decision Tree
dt_feats, dt_scores, dt_best_set, dt_best_score = forward_feature_selection(X_train_df, y_train, DecisionTreeRegressor(random_state=random_state),scoring='neg_mean_absolute_error', cv= 5, verbose=True)

# Feature Selection for Random Forest (Feature Importance)
rf_temp = RandomForestRegressor(random_state=random_state, n_jobs=-1)
rf_temp.fit(X_train_df, y_train)

Added Best Feature: finishedsquarefeet12, CV Score (MAE): 184449.8978
Added Best Feature: latitude, CV Score (MAE): 178219.5539
Added Best Feature: longitude, CV Score (MAE): 175707.9634
Added Best Feature: bath_per_bed, CV Score (MAE): 174019.9393
Added Best Feature: roomcnt, CV Score (MAE): 173511.8663
Added Best Feature: yearbuilt, CV Score (MAE): 173155.9786
Added Best Feature: calculatedfinishedsquarefeet, CV Score (MAE): 172715.4057
Added Best Feature: garagetotalsqft, CV Score (MAE): 172401.7441
Added Best Feature: regionidcounty, CV Score (MAE): 172052.4827
Added Best Feature: bedroomcnt, CV Score (MAE): 171865.8121
Added Best Feature: regionidneighborhood, CV Score (MAE): 171701.7554
Added Best Feature: bathroomcnt, CV Score (MAE): 171618.5398
Added Best Feature: fullbathcnt, CV Score (MAE): 171476.6958
Added Best Feature: lotsizesquarefeet, CV Score (MAE): 171398.9479
Added Best Feature: buildingqualitytypeid, CV Score (MAE): 171312.4976
Added Best Feature: regionidcity, CV S

RandomForestRegressor(n_jobs=-1, random_state=42)

In [ ]:
# Identify the best-performing subset of features
importances = rf_temp.feature_importances_
rf_best_set = X_train_df.columns[importances > np.mean(importances)].tolist()
print(f"Best-Performing Subset of Features: {rf_best_set}")

Best-Performing Subset of Features: ['calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'latitude', 'longitude', 'lotsizesquarefeet', 'regionidzip', 'yearbuilt', 'log_sqft']


In [ ]:
# Re-run models
final_results = {'Linear Regression': (LinearRegression(), lr_best_set),
                 'Decision Tree': (DecisionTreeRegressor(random_state=random_state), dt_best_set),
                 'Random Forest': (RandomForestRegressor(random_state=random_state, n_jobs=-1), rf_best_set)}

print("Mean & Standard Deviation of CV MAE Scores")

for name, (model, features) in final_results.items():

    X_subset = X_train_df[features]

    scores = cross_val_score(model, X_subset, y_train, cv = cv,scoring ='neg_mean_absolute_error', n_jobs =-1)
    mae = -scores
    print(f"Model: {name}")
    print(f"Features Used ({len(features)}): {features}")
    print(f"Mean MAE: ${mae.mean():,.2f}")
    print(f"Std MAE:  ${mae.std():,.2f}\n")

Mean & Standard Deviation of CV MAE Scores
Model: Linear Regression
Features Used (18): ['finishedsquarefeet12', 'latitude', 'longitude', 'bath_per_bed', 'roomcnt', 'yearbuilt', 'calculatedfinishedsquarefeet', 'garagetotalsqft', 'regionidcounty', 'bedroomcnt', 'regionidneighborhood', 'bathroomcnt', 'fullbathcnt', 'lotsizesquarefeet', 'buildingqualitytypeid', 'regionidcity', 'garagecarcnt', 'propertylandusetypeid']
Mean MAE: $171,271.50
Std MAE:  $1,017.18

Model: Decision Tree
Features Used (3): ['regionidzip', 'bathroomcnt', 'propertylandusetypeid']
Mean MAE: $163,958.30
Std MAE:  $1,204.40



/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Model: Random Forest
Features Used (8): ['calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'latitude', 'longitude', 'lotsizesquarefeet', 'regionidzip', 'yearbuilt', 'log_sqft']
Mean MAE: $152,180.55
Std MAE:  $940.06



### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


Feature selection had mixed but mostly positive effects depending on the model.

The decision tree model improved the most. 	In Part 1, its MAE was 196,707, to 197,204 in part 2, but dropped sharply to 163,958 in Part 3 with only three features: `regionidzip`, `bathroomcnt`, and `propertylandusetypeid`. The model reduced overfitting from unnecessary splits.

The linear regression model fluctuated from a MAE of 171,387, to 171,438, and then to 171,272 in Part 3. Removing highly collinear features helped slightly, but the model was already stable.

The random forest modeldid not benefit from feature selection. Its MAE slightly worsened from 151,411 (Part 1) to 152,181 (Part 3), which is expected since Random Forest already performs internal feature selection. Overall, Random Forest remained the best-performing model. The most important features across models were `latitude`, `longitude`, `regionidzip`, and `propertysize`, while engineered features contributed little new information.




### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above.
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks.
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [6]:
# Add as many cells as you need

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

# Ridge Regression

X_ridge = X_train_df[lr_best_set]
ridge_params = {'alpha': [0.01, 0.1, 1, 10, 100, 500, 1000]}
ridge_search = GridSearchCV(Ridge(), ridge_params, scoring='neg_mean_absolute_error', cv = 5, n_jobs =-1)
ridge_search.fit(X_ridge, y_train)

best_ridge = ridge_search.best_estimator_
ridge_scores = cross_val_score(best_ridge, X_ridge, y_train, cv = cv, scoring='neg_mean_absolute_error', n_jobs=-1)

# Decision Tree

X_dt = X_train_df[dt_best_set]
dt_params = {
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
dt_search = GridSearchCV(DecisionTreeRegressor(random_state=random_state), dt_params, scoring='neg_mean_absolute_error', cv= 5, n_jobs = -1)
dt_search.fit(X_dt, y_train)

best_dt = dt_search.best_estimator_
dt_scores = cross_val_score(best_dt, X_dt, y_train, cv = cv, scoring='neg_mean_absolute_error', n_jobs= -1)

# Random Forest

X_rf = X_train_df[rf_best_set]
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'max_features': ['sqrt', 'log2', None]
}
rf_search = GridSearchCV(RandomForestRegressor(random_state=random_state, n_jobs=-1), rf_params, scoring='neg_mean_absolute_error', cv = 5, n_jobs = -1)
rf_search.fit(X_rf, y_train)

best_rf = rf_search.best_estimator_
rf_scores = cross_val_score(best_rf, X_rf, y_train, cv = cv, scoring='neg_mean_absolute_error', n_jobs=-1)



NameError: name 'X_train_df' is not defined

In [ ]:
results = [("Ridge Regression", ridge_search.best_params_, -ridge_scores),("Decision Tree", dt_search.best_params_, -dt_scores),("Random Forest", rf_search.best_params_, -rf_scores)]

for name, params, mae_array in results:
    print(f"{name})
    print(f"Best Params: {params}")
    print(f"Mean CV MAE: ${mae_array.mean():,.2f}")
    print(f"Std CV MAE:  ${mae_array.std():,.2f}")
    print()


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


> Your text here

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants.

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set.




In [ ]:
# Add as many cells as you need


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

> Your text here